# 9 — Analysis

**The concept:** `analyze`, `validate` and `explain` tell you what a solve *would* do, reading only
signatures, annotations and the dependency graph.

**No discipline is ever called.** That is the invariant the whole layer rests on: introspection
never requires execution. It also means these are free, so there is no reason not to run them.

In [1]:
from smartmdao import Pipeline, HybridSolver, analyze, validate, explain, PipelineAnalysis

pipeline = Pipeline(solver=HybridSolver())

@pipeline.step(outputs=["y1"])
def discipline_1(z: float, y2: float) -> float:
    return z**2 - 0.2 * y2

@pipeline.step(outputs=["y2"])
def discipline_2(y1: float) -> float:
    return abs(y1) ** 0.5

@pipeline.step(outputs=["objective"])
def objective(y1: float, y2: float) -> float:
    return y1 + y2

report = analyze(pipeline, inputs=["z"])
print(type(report).__name__)
print()
print("steps:            ", report.steps)
print("execution order:  ", report.execution_order)
print("external inputs:  ", report.external_inputs)
print("terminal outputs: ", report.terminal_outputs)
print("has cycles:       ", report.has_cycles)
print("recommended:      ", report.recommended_solver)
print("because:          ", report.reason)

PipelineAnalysis

steps:             ('discipline_1', 'discipline_2', 'objective')
execution order:   ('discipline_1', 'discipline_2', 'objective')
external inputs:   ('z',)
terminal outputs:  ('objective',)
has cycles:        True
recommended:       HybridSolver
because:           1 feedback loop(s) detected; DAGSolver would raise. HybridSolver runs acyclic steps once and iterates only the cyclic blocks.


## The analysis is **solver-aware**

This is not a detail. `IterativeSolver` sweeps in registration order and ignores the graph;
`HybridSolver` follows the graph and runs cyclic blocks alphabetically. **The same steps need
different variables seeded**, so an analysis that gave one answer for every pipeline would be
wrong.

In [2]:
from smartmdao import IterativeSolver, CycleAnalysis

hybrid_view = analyze(pipeline, inputs=["z"])

iterative = Pipeline(solver=IterativeSolver())
for fn, outs in ((discipline_1, ["y1"]), (discipline_2, ["y2"]), (objective, ["objective"])):
    iterative.add(fn, outputs=outs)
iterative_view = analyze(iterative, inputs=["z"])

print("HybridSolver    order:", hybrid_view.execution_order)
print("IterativeSolver order:", iterative_view.execution_order)
print()
for cycle in hybrid_view.cycles:
    print("cycle:", cycle.steps, "coupling on", cycle.feedback_variables)
    print("is a CycleAnalysis:", isinstance(cycle, CycleAnalysis))

HybridSolver    order: ('discipline_1', 'discipline_2', 'objective')
IterativeSolver order: ('discipline_1', 'discipline_2', 'objective')

cycle: ('discipline_1', 'discipline_2') coupling on ('y1', 'y2')
is a CycleAnalysis: True


## `validate()` — every structural problem at once

`validate()` never raises and never stops at the first problem. It returns a tuple of `Finding`,
worst first, so a whole pipeline can be fixed in one pass.

In [3]:
from smartmdao import Finding

print("a healthy pipeline:", validate(pipeline, inputs=["z"]))

a healthy pipeline: (Finding(code='initial-guess-required', severity='error', message="'discipline_1' consumes 'y2' before anything produces it, so the first sweep has nothing to read. Ordering here is decided by HybridSolver, so renaming a step or swapping solvers can change which variable this is. Pass y2=... to run().", step='discipline_1', variable='y2'),)


In [4]:
broken = Pipeline()

@broken.step(outputs=["mass"])
def structures(span: float) -> float:
    return span * 12.0

@broken.step(outputs=["mass"])                 # duplicate output
def alternative_mass(span: float) -> str:
    return "heavy"

@broken.step(outputs=["cost"])
def costing(mass: float, rate: float) -> float:   # `rate` comes from nowhere
    return mass * rate

for finding in validate(broken, inputs=["span"]):
    print(f"[{finding.severity.upper():7}] {finding.code}")
    print(f"          {finding.message}")
    print()

[ERROR  ] duplicate-output
          'mass' is declared as an output by both 'structures' and 'alternative_mass'. Only 'alternative_mass' will be wired up; the other result is computed and discarded.

[ERROR  ] type-mismatch
          'alternative_mass' declares mass -> str, but 'costing' expects mass: float.

[ERROR  ] missing-input
          'rate' is required by 'costing' but no step produces it and it was not listed as an input.



Each `Finding` carries a machine-readable `code`, a `severity`, and where it applies — so
tooling can act on it while a human reads the message.

In [5]:
finding = validate(broken, inputs=["span"])[0]
print("code:    ", finding.code)
print("severity:", finding.severity)
print("step:    ", finding.step)
print("variable:", finding.variable)
print()
print("rendered:", str(finding))
print("is a Finding:", isinstance(finding, Finding))

code:     duplicate-output
severity: error
step:     alternative_mass
variable: mass

rendered: ERROR: 'mass' is declared as an output by both 'structures' and 'alternative_mass'. Only 'alternative_mass' will be wired up; the other result is computed and discarded. [alternative_mass]
is a Finding: True


## The expensive mistake: a disconnected graph

This is the one worth knowing about. A pipeline whose graph falls into **separate pieces** validates
clean on every other check, converges, computes correct arithmetic — and produces an answer that
ignores half its inputs.

It was found in real use, where a generated wing model ignored three of its five inputs.

In [6]:
disconnected = Pipeline(solver=HybridSolver())

@disconnected.step(outputs=["lift"])
def compute_lift(span: float, chord: float, speed: float) -> float:
    return 0.5 * 1.225 * speed**2 * span * chord     # nothing ever reads `lift`

@disconnected.step(outputs=["mass"])
def mass_loop(mass: float) -> float:
    return 100.0 + 0.05 * mass                        # a loop closed on itself

for finding in validate(disconnected, inputs=["span", "chord", "speed", "mass"]):
    print(f"[{finding.code}] {finding.message}")

[disconnected-graph] The pipeline falls into 2 disconnected pieces, so nothing computed in one can affect another: ['compute_lift'] -> ['lift'] consumed by nothing; ['mass_loop']. This usually means a discipline was never wired in - its inputs will have no influence on the result, however well the rest converges.


The arithmetic is right. The convergence is real. The answer does not depend on span, chord
or speed at all — and only the *disconnection* reveals it.

## `explain()` — the same thing, in prose

For a human, a review, or an agent that would rather read a paragraph than a data structure.

In [7]:
print(explain(pipeline, inputs=["z"]))

Pipeline with 3 step(s): discipline_1, discipline_2, objective.

Recommended solver: HybridSolver
  1 feedback loop(s) detected; DAGSolver would raise. HybridSolver runs acyclic steps once and iterates only the cyclic blocks.

External inputs: z
Final outputs:   objective

Feedback loops (1):
  1. discipline_1 -> discipline_2
     coupling on: y1, y2

Needs initial values for: y2 (for discipline_1)

Execution order: discipline_1 -> discipline_2 -> objective

Findings (1):
  ERROR: 'discipline_1' consumes 'y2' before anything produces it, so the first sweep has nothing to read. Ordering here is decided by HybridSolver, so renaming a step or swapping solvers can change which variable this is. Pass y2=... to run(). [discipline_1]


## Analysis is the cheapest rung on the cost ladder

| Rung | Cost |
|---|---|
| `analyze` / `validate` / `explain` | **free** — nothing executes |
| one sweep | one call per discipline |
| budgeted run | capped sweeps and a wall clock |
| full run | whatever it takes |
| `optimize` | a full run, 10²–10³ times |

Everything on this page sits on the top row. Run it always; it costs nothing and it catches the
class of mistake that otherwise reaches a design review.

---

**Next:** [10 — Visualization](10-visualization.ipynb).